# Processamento Avançado de Dados e Engenharia de Atributos: Dataset ESAVI

O objetivo deste script é realizar uma **limpeza estatística radical** no meu conjunto de dados brutos de ESAVI. 
Seguindo os preceitos teóricos de análise multivariada, o meu foco aqui é a transformação de dados textuais e cronológicos complexos em um **Espaço de Características (Feature Space)** puramente numérico e geométrico para que eu possa aplicar modelos estatísticos avançados.

### O que eu resolvo com este código:
1. **Eliminação de Ruído:** Descarte definitivo de colunas descritivas livres, redundantes ou com dados massivamente ausentes (*Sparsity Extrema*).
2. **Explosão Cronológica Cruzada:** Separação das doses aplicadas e cruzamento imediato com o tipo de evento (Erro vs Evento Adverso) para capturar a cronologia dos fatos.
3. **Binarização (Dummies):** Transformação de categorias qualitativas (Sexo, Raça) em eixos ortogonais binários.
4. **Padronização Geométrica:** Escalonamento (*StandardScaler*) para blindar as minhas análises de Covariância, Heatmaps e PCA contra distorções de magnitude.

In [14]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler

# ==============================================================================
# 1. CARREGAMENTO DOS DADOS ORIGINAIS
# ==============================================================================
print("Lendo o arquivo original sem limpezas prévias...")
# Carrego o arquivo bruto enviado para iniciar o meu pipeline de tratamento
df_original = pd.read_csv('Planilha todas as cidade ESAVI.xlsx - Página1.csv')

Lendo o arquivo original sem limpezas prévias...


## Etapa 1: Eliminação de Colunas Inúteis para Análise Multivariada

Nesta etapa, fiz a filtragem do dataset mantendo apenas o cerne matemático necessário para as minhas análises. Eliminei permanentemente textos livres de sintomas ou medicamentos para evitar esparsidade extrema, descartei dados com alta ausência de registro para não zerar a variância e removi identificadores geográficos ou datas que se comportavam apenas como ruído não linear.

In [15]:
# Defino rigidamente as colunas que carregam a variância útil para a minha modelagem
cols_uteis = [
    'Idade Evento', 'Dose', 'Tipo de Evento', 'Sexo', 'Raça/Cor', 
    'Classificação de gravidade', 'Houve atendimento médico?'
]
df_filtrado = df_original[cols_uteis].copy()

# Crio o meu DataFrame que vai receber as novas dimensões puras
df_limpo = pd.DataFrame()

# Trato a Idade: removo textos e forço o tipo numérico contínuo preenchendo falhas com zero
df_limpo['Idade'] = pd.to_numeric(df_filtrado['Idade Evento'], errors='coerce').fillna(0)

## Etapa 2: Explosão Cronológica Cruzada [Dose X Tipo de Evento]

A coluna original de `Dose` misturava múltiplos históricos textuais. Para permitir que o meu modelo entenda **quando** e **qual** tipo de evento ocorreu, criei **10 novas colunas booleanas independentes**. 
Cada dose (de 1 a 5) passa a ser avaliada por mim de forma cruzada com o tipo de notificação (Erro operacional vs Evento Adverso biológico).

In [16]:
# Crio uma função para isolar cada dose (1 a 5) e mapear se o evento foi Erro ou Evento Adverso
def extrair_ocorrencias_por_dose(row):
    texto_dose = str(row['Dose']).lower()
    texto_evento = str(row['Tipo de Evento']).lower()
    
    flags = {}
    # Faço um loop para cobrir as 5 doses possíveis registradas nas colunas textuais
    for d in range(1, 6):
        # Crio um padrão de busca para capturar tanto "1" quanto "1ª" no meu texto limpo
        alvo_dose = f'{d}ª' if f'{d}ª' in texto_dose else str(d)
        
        # Atribuo 1 apenas se o número da dose existe na célula E o tipo de evento bate com o meu critério
        flags[f'Erro_Dose_{d}'] = 1 if (alvo_dose in texto_dose and 'erro' in texto_evento) else 0
        flags[f'Adverso_Dose_{d}'] = 1 if (alvo_dose in texto_dose and 'adverso' in texto_evento) else 0
        
    return pd.Series(flags)

print("Executando a explosão cruzada das doses...")
flags_cronologicas = df_filtrado.apply(extrair_ocorrencias_por_dose, axis=1)
df_limpo = pd.concat([df_limpo, flags_cronologicas], axis=1)

Executando a explosão cruzada das doses...


## Etapa 3: Codificação Categórica e Definição de Variáveis Alvo

* **Sexo e Raça/Cor:** Converti estas informações em colunas binárias (*One-Hot Encoding/Dummies*). Geometricamente, isso transforma uma categoria textual em um novo eixo ortogonal no meu espaço.
* **Classificação de Gravidade:** Converti para uma escala **Ordinal** (0 = Não grave, 1 = Hospitalização, 2 = Óbito). Isso dá o peso hierárquico correto para o cálculo de correlação do meu ecossistema.
* **Atendimento Médico:** Binarizei em `1` (Sim) e `0` (Não) para atuar como a minha variável resposta primária.

In [17]:
# Trato os nulos textuais antes de gerar as colunas binárias
df_filtrado['Sexo'] = df_filtrado['Sexo'].str.strip().fillna('Nao_Informado')
df_filtrado['Raça/Cor'] = df_filtrado['Raça/Cor'].str.strip().fillna('Nao_Informado')

# Transformo categorias de texto em novos eixos ortogonais binários (0 ou 1)
dummies = pd.get_dummies(df_filtrado[['Sexo', 'Raça/Cor']], prefix=['Sexo', 'Raca']).astype(int)
df_limpo = pd.concat([df_limpo, dummies], axis=1)

# Transformo a gravidade textual longa em uma escala numérica de peso crescente (Corrigido para Nivel_Gravidade)
mapa_gravidade = {'1: Não grave': 0, '1: Requer hospitalização': 1, '1: Obito': 2}
df_limpo['Nivel_Gravidade'] = df_filtrado['Classificação de gravidade'].map(mapa_gravidade).fillna(0).astype(int)

# Binarizo o desfecho médico (1 = Sim, 0 = Não)
df_limpo['Atendimento_Medico'] = (df_filtrado['Houve atendimento médico?'].str.strip() == 'Sim').astype(int)

# Removo linhas que ficaram duplicadas após a simplificação e retirada do ruído textual
df_limpo = df_limpo.drop_duplicates().reset_index(drop=True)

# EXPORTAÇÃO 1: Salvo o meu arquivo contável/binário puro
nome_arquivo_puro = 'dataset_ESAVI_totalmente_limpo.csv'
df_limpo.to_csv(nome_arquivo_puro, index=False)
print(f"Meu arquivo contável foi gerado com sucesso: '{nome_arquivo_puro}'")

Meu arquivo contável foi gerado com sucesso: 'dataset_ESAVI_totalmente_limpo.csv'


## Etapa 4: Padronização Z-Score para PCA e Covariância

Algoritmos multidimensionais como o PCA baseiam-se na distância geométrica dos dados. Se eu mantivesse a variável `Idade` variando de 0 a 90 ao lado de colunas binárias que variam apenas de 0 a 1, a Idade distorceria os eixos, dominando a variância artificialmente.

Utilizo o `StandardScaler` para forçar todas as variáveis a terem **Média = 0** e **Desvio Padrão = 1**, deixando o dataset perfeitamente equilibrado e pronto para as minhas análises matemáticas posteriores.

In [18]:
# Aplico a padronização Z-Score para gerar a minha base matemática pura
# Isso impede que a magnitude da Idade mascare as minhas variáveis binárias
scaler = StandardScaler()
dados_escalonados = scaler.fit_transform(df_limpo)
df_scaled = pd.DataFrame(dados_escalonados, columns=df_limpo.columns)

# EXPORTAÇÃO 2: Salvo o meu arquivo estruturado e pronto para PCA e Heatmaps
nome_arquivo_scaled = 'dataset_ESAVI_pronto_para_PCA_e_Heatmap.csv'
df_scaled.to_csv(nome_arquivo_scaled, index=False)
print(f"Meu arquivo escalonado foi gerado com sucesso: '{nome_arquivo_scaled}'")

# --- RELATÓRIO DO MEU NOVO ESPAÇO DE CARACTERÍSTICAS ---
print("\n" + "="*50)
print("MEU DIAGNÓSTICO FINAL DO DATASET MULTIVARIADO")
print("="*50)
print(f"Registros únicos que obtive (Linhas): {df_limpo.shape[0]}")
print(f"Características puras que gerei (Colunas): {df_limpo.shape[1]}")
print("\nTodas as minhas colunas atuais estão numéricas e validadas:")
print(list(df_limpo.columns))

Meu arquivo escalonado foi gerado com sucesso: 'dataset_ESAVI_pronto_para_PCA_e_Heatmap.csv'

MEU DIAGNÓSTICO FINAL DO DATASET MULTIVARIADO
Registros únicos que obtive (Linhas): 386
Características puras que gerei (Colunas): 23

Todas as minhas colunas atuais estão numéricas e validadas:
['Idade', 'Erro_Dose_1', 'Adverso_Dose_1', 'Erro_Dose_2', 'Adverso_Dose_2', 'Erro_Dose_3', 'Adverso_Dose_3', 'Erro_Dose_4', 'Adverso_Dose_4', 'Erro_Dose_5', 'Adverso_Dose_5', 'Sexo_Feminino', 'Sexo_Masculino', 'Sexo_Nao_Informado', 'Raca_0', 'Raca_Amarela', 'Raca_Branca', 'Raca_Nao_Informado', 'Raca_Negra', 'Raca_Parda', 'Raca_Preta', 'Nivel_Gravidade', 'Atendimento_Medico']
